# Hindi / Hinglish intent classifier: train + UI on Colab (T4)

Run the cells **in order**. Each step says what to look for before moving on.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU** > Save.
2. In your Google Drive, create a folder called `nlu` and put these 6 files in it:
   `nlu_pipeline.py`, `nlu_infer.py`, `train_final.py`, `app.py`, `train.csv`, `test.csv`

## Step 1: Check the GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then re-run.")

## Step 2: Mount Drive and check your files
Allow access when the pop-up asks. Change `PROJECT` if your folder has a different name.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = "/content/drive/MyDrive/nlu"
need = ["nlu_pipeline.py", "nlu_infer.py", "train_final.py", "app.py", "train.csv", "test.csv"]
missing = [f for f in need if not os.path.exists(os.path.join(PROJECT, f))]
print("Missing files:", missing if missing else "none, all good")
assert not missing, "Upload the missing files to the Drive folder first."
%cd $PROJECT

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

need = [
    "nlu_pipeline.py",
    "nlu_infer.py",
    "train_final.py",
    "app.py",
    "train.csv",
    "test.csv"
]

missing = [f for f in need if not os.path.exists(f)]

print("Missing files:", missing if missing else "All files uploaded!")

## Step 3: Install the extra packages
(torch, transformers, scikit-learn and pandas are already on Colab.) Takes about a minute.

In [ ]:
!pip install -q -U "gradio>=6" indic-transliteration

## Step 4: Quick smoke test (about 3-5 minutes)
Trains on 3,000 rows for 1 epoch. This downloads MuRIL (~950 MB) and proves the whole pipeline works on your GPU **before** you spend 30+ minutes on the real run. The score here will be low; that is expected.

Look for: `Transformer (holdout)`, `parameters: ... (limit 300M)`, `saved bundle reloaded`, `wrote ... submission_smoke.csv`.
If MuRIL cannot be downloaded, add `--model xlm-roberta-base` to this command and to the Step 5 command.

In [ ]:
!python train_final.py --data_dir . --out_dir /content/smoke --smoke --epochs 1

## Step 5: Full training (roughly 20-40 minutes, estimated)
Trains on 90% of the data, tunes the blend on the 10% holdout, then retrains on 100% (`--full_refit`). Output goes to Drive, so the model survives a disconnect.

**What good looks like in the log**
- `SVM (holdout)` should be about 0.83.
- `Transformer (holdout)` should be at or above the SVM. If it is clearly below, try `--lr 2e-5` or `--epochs 6`.
- `blend weight on transformer` shows how much the ensemble leans on MuRIL.
- `parameters: ... = ...M (limit 300M)` must be under 300M (expect about 244M).
- The final `saved bundle reloaded: holdout macro-F1` is optimistic with `--full_refit` (holdout rows were in training). Use the earlier `holdout` lines as your honest estimate.

Keep this browser tab open and the session connected while it runs.
If you run out of GPU memory, add `--bs 16`.

In [ ]:
!python train_final.py --data_dir . --out_dir "$PROJECT/final" --full_refit

## Step 6: Check and download the submission file

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if "submission" in file.lower() or file.endswith(".csv"):
            print(os.path.join(root, file))

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
%cd /content
!python train_final.py

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if "submission" in file.lower():
            print(os.path.join(root, file))

In [ ]:
import pandas as pd

sub = pd.read_csv("/content/final/submission.csv")
test = pd.read_csv("/content/test.csv")

print("Submission shape:", sub.shape)
print("Columns:", sub.columns.tolist())

assert len(sub) == len(test), "Row count mismatch!"
assert (sub.example_id.values == test.example_id.values).all(), "ID order mismatch!"
assert sub.target.between(0, 55).all(), "Invalid class labels!"

print("Submission format is valid!")
print("Class 55 share: %.1f%%" % (100 * (sub.target == 55).mean()))

display(sub.head())

In [ ]:
from google.colab import files

files.download("/content/final/submission.csv")

## Step 7: Launch the UI
The cell keeps running while the app is up. Look for the line **`Running on public URL: https://....gradio.live`** and open it in a new tab.
The link works only while this cell runs (and for up to one week). Stop the cell to shut the UI down.

In [ ]:
!python app.py --bundle "$PROJECT/final/bundle" --share

## Optional: use the model from Python (no UI)
Run this in a new cell after Step 5 (stop the Step 7 cell first, or run it in a separate session).

In [ ]:
import sys; sys.path.insert(0, PROJECT)
from nlu_infer import IntentClassifier
clf = IntentClassifier(f"{PROJECT}/final/bundle")
for r in clf.predict(["kripaya rasoeeghar ki battiyaan band kar den", "आज मौसम कैसा है"], topk=3):
    print(r["text"], "->", clf.name(r["pred"]), f"{r['confidence']:.1%}", "| top3:", r["top"])

## Troubleshooting
| Problem | Fix |
|---|---|
| `ModuleNotFoundError: nlu_pipeline` | Re-run Step 2 so `%cd` puts you in the project folder. |
| CUDA out of memory | Add `--bs 16` to the train command. |
| MuRIL download fails | Add `--model xlm-roberta-base` (about 278M params, still under the limit). |
| Session disconnected mid-training | Reconnect, re-run Steps 1-3, then Step 5 again (training restarts; files already on Drive are kept). |
| No `gradio.live` link | Re-run Step 3, then Step 7. |